In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split ,cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
                    accuracy_score, precision_score, recall_score, f1_score, 
                    confusion_matrix, classification_report, roc_curve, auc)
# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Utility Function
#from utils import *

sns.set_theme(style='whitegrid')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [2]:
df= pd.read_csv(r'C:\Users\user\Jupyter NoteBook\Titanic Survival predictioin\titanic (2)\train_cleaned.csv')
print(f"Data set shape {df.shape}")
df.head()

Data set shape (891, 23)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,HasCabin,FamilySize,IsAlone,...,Title_Mr,Title_Mrs,Title_Rare,AgeGroup_Teen,AgeGroup_Young Adult,AgeGroup_Adult,AgeGroup_Senior,FareBin_Medium,FareBin_High,FareBin_Very High
0,0,3,0,22.0,1,0,7.2500,0,2,0,...,1,0,0,0,1,0,0,0,0,0
1,1,1,1,38.0,1,0,71.2833,1,2,0,...,0,1,0,0,0,1,0,0,0,1
2,1,3,1,26.0,0,0,7.9250,0,1,1,...,0,0,0,0,1,0,0,1,0,0
3,1,1,1,35.0,1,0,53.1000,1,2,0,...,0,1,0,0,1,0,0,0,0,1
4,0,3,0,35.0,0,0,8.0500,0,1,1,...,1,0,0,0,1,0,0,1,0,0


In [3]:
X = df.drop(columns='Survived')
y = df["Survived"]


print(f"Shape of X -- {X.shape}")
print(y.value_counts())
print(f"Shape of y -- {y.shape}")

Shape of X -- (891, 22)
Survived
0    549
1    342
Name: count, dtype: int64
Shape of y -- (891,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y ,
                                                    test_size=0.2, random_state= 42, stratify = y)
print(f"Train Set has - {X_train.shape[0]} samples")
print(f"Test Set has - {X_test.shape[0]} samples" )
print(f"\nTrain Survival rate {y_train.mean():.2%}")
print(f"Test Survival rate {y_test.mean():.2%}")

Train Set has - 712 samples
Test Set has - 179 samples

Train Survival rate 38.34%
Test Survival rate 38.55%


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns= X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

X_train_scaled.head(3)

,Pclass,Sex,Age,SibSp,Parch,Fare,HasCabin,FamilySize,IsAlone,Embarked_Q,...,Title_Mr,Title_Mrs,Title_Rare,AgeGroup_Teen,AgeGroup_Young Adult,AgeGroup_Adult,AgeGroup_Senior,FareBin_Medium,FareBin_High,FareBin_Very High
692,0.829568,-0.742427,-0.322182,-0.465084,-0.466183,0.513812,-0.538382,-0.556339,0.800346,-0.289333,...,0.85332,-0.420547,-0.174329,-0.283593,0.875642,-0.592489,-0.15162,-0.583838,-0.570863,1.745123
481,-0.370945,-0.742427,0.053575,-0.465084,-0.466183,-0.662563,-0.538382,-0.556339,0.800346,-0.289333,...,0.85332,-0.420547,-0.174329,-0.283593,0.875642,-0.592489,-0.15162,-0.583838,-0.570863,-0.573025
527,-1.571457,-0.742427,0.805089,-0.465084,-0.466183,3.955399,1.857418,-0.556339,0.800346,-0.289333,...,0.85332,-0.420547,-0.174329,-0.283593,-1.142019,1.687794,-0.15162,-0.583838,-0.570863,1.745123


In [6]:
def evaluate_model(y_true, y_pred, model_name="Model"):
    metric = {
        "Model": model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
    }
    print(f"\n{'='*100}")
    print(f"\n{'='*50}")
    print(f"    {model_name} Results")
    print(f"{'='*50}")

    print(f"    Accuracy:    {metric['Accuracy']:.4f}")
    print(f"    Precision:   {metric["Precision"]:.4f}")
    print(f"    Recall:      {metric['Recall']:.4f}")
    print(f"    F1 Score:    {metric['F1 Score']:.4f}  ")

    print(f"{'='*50}")
    print("\nClassification Report:")
    print(classification_report(y_true ,y_pred))
    print(f"\n{'='*100}")
    return metric

def plot_confusion_matrics(y_true, y_pred, model_name="Model"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,4))
    sns.heatmap(
        cm,
        cmap='Blues',
        annot =True,
        fmt = 'd',
        xticklabels=['Did Not Survived', 'Did Survived'],
        yticklabels=['Did Not Survived', 'Did Survived'],
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()


In [7]:
# store result for comparism
results = []
trained_model = {}

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score


# -----------------------------
# MODELING
# -----------------------------
rf= RandomForestClassifier(random_state =42, n_jobs=-1)

svm_pipe = Pipeline([('scaler' , StandardScaler()),
                     ('svc', SVC(random_state=42,probability= True))])

knn =  Pipeline([('scaler' , StandardScaler()),
                 ('KNN', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))])

gb =GradientBoostingClassifier(random_state= 42)

lr = Pipeline([
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(random_state=42, n_jobs=-1))])



# -----------------------------
# PARAM
# -----------------------------

rf_param = {
    'criterion': ['gini', 'entropy'],
    'max_features' : ['sqrt', 'log2', None],
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}
# svm_param = {
#     'svc__C': [0.1, 1, 10],
#     'svc__kernel': ['linear', 'rbf'],
#     'svc__gamma': ['scale'],
#     'svc__class_weight': [None, 'balanced']
# }
knn_param = {
    'KNN__n_neighbors' : [3, 5, 7, 9 ,10 ,13, 15, 18],
    'KNN__weights' : ['uniform', 'distance'],
    'KNN__algorithm' : ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'KNN__p': [1, 2]
}

gb_param = {
    'loss' : ['log_loss', 'exponential'],
    'learning_rate' : [0.1, 0.01, 0.05],
    'n_estimators' : [100, 200 ,300],
    'max_depth' : [3 ,5],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'subsample': [0.8, 1.0],
}
lr_param = {
    'lr__C': [0.01, 0.1, 1, 10, 100],
    'lr__penalty': ['l1', 'l2'],
    'lr__solver': ['liblinear', 'lbfgs'],
    'lr__class_weight': [None, 'balanced'],
}
# -----------------------------
# GRID SEARCH
# -----------------------------

rf_grid = GridSearchCV(rf, 
                    param_grid=rf_param,
                    cv=3,
                    scoring= {'accuracy': 'accuracy', 'f1': 'f1', 'recall' :'recall', 'precision': 'precision'},
                    refit='accuracy',
                    n_jobs=-1,
                    verbose=5,
)
# svm_grid = GridSearchCV(svm_pipe,
#                     param_grid = svm_param,
#                     cv =3,
#                     scoring= {'accuracy': 'accuracy', 'f1': 'f1', 'recall' :'recall', 'precision': 'precision'},
#                     refit='accuracy',
#                     n_jobs = -1,
#                     verbose= 5,
# )
knn_grid = GridSearchCV(
                    knn,
                    param_grid= knn_param,
                    cv = 3,
                    scoring= {'accuracy': 'accuracy', 'f1': 'f1', 'recall' :'recall', 'precision': 'precision'},
                    refit= 'accuracy',
                    verbose= 5,
                    n_jobs=-1
)
gb_grid  =GridSearchCV(gb,
                    param_grid= gb_param,
                    cv = 3,
                    scoring= {'accuracy': 'accuracy', 'f1': 'f1', 'recall' :'recall', 'precision': 'precision'},
                    refit= 'accuracy',
                    verbose= 5,
                    n_jobs=-1
)
lr_grid = GridSearchCV(
                    lr,
                    param_grid= lr_param,
                    cv = 3,
                    scoring= {'accuracy': 'accuracy', 'f1': 'f1', 'recall' :'recall', 'precision': 'precision'},
                    refit= 'accuracy',
                    verbose= 5,
                    n_jobs=-1,
)


# -----------------------------
# FITTING 
# -----------------------------
rf_grid.fit(X_train, y_train)
print(rf_grid.best_score_)
print(rf_grid.best_estimator_)
# svm_grid.fit(X_train, y_train)
# print(svm_grid.best_score_)
# print(svm_grid.best_estimator_)
knn_grid.fit(X_train, y_train)
print(knn_grid.best_score_)
print(knn_grid.best_estimator_)
gb_grid.fit(X_train, y_train)
print(gb_grid.best_score_)
print(gb_grid.best_estimator_)
lr_grid.fit(X_train, y_train)
print(lr_grid.best_score_)
print(lr_grid.best_estimator_)


Fitting 3 folds for each of 216 candidates, totalling 648 fits
0.8455069791629732
RandomForestClassifier(criterion='entropy', max_depth=10, max_features=None,
                       min_samples_leaf=2, min_samples_split=5, n_jobs=-1,
                       random_state=42)
Fitting 3 folds for each of 128 candidates, totalling 384 fits
0.8230625583566761
Pipeline(steps=[('scaler', StandardScaler()),
                ('KNN',
                 KNeighborsClassifier(algorithm='ball_tree', n_jobs=-1,
                                      n_neighbors=13, p=1))])
Fitting 3 folds for each of 288 candidates, totalling 864 fits
0.8356735099102933
GradientBoostingClassifier(learning_rate=0.01, max_depth=5, min_samples_split=5,
                           n_estimators=200, random_state=42, subsample=0.8)
Fitting 3 folds for each of 40 candidates, totalling 120 fits
0.8230330106726235
Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(C=1, n_jobs=-1

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier

#print("XGBoost version:", xgb.__version__)
xgbc = XGBClassifier()